# Fraud Transaction Detection - Business case

## 1. Business Understanding

The objective of this project is to develop a machine learning model to predict fraudulent financial transactions using historical transaction data.  
The model aims to help the organization reduce financial losses caused by fraud while minimizing false positives that may impact genuine customers.


## 2. problem statement

Given transaction level data, is to build the binary classification model to predict whether the trancation is fradulent or non fradulent

## 3. Evalvation metrics

since the fradulent transctions are typically rare, the datasets are expected to be highly imbalanced. model performance will be evalvated using Recall, F1 score, precision, RUC-AUC and the Confusion matrix, with the primary focus on Recall maximizing the fradulent transactions.

In [ ]:
import pandas as pd

In [ ]:
import pandas as pd

df = pd.read_csv("Fraud.csv")
df.head(5)


In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df['isFraud'].value_counts()

### Target variable distribution
The dataset is highly imbalanced, with fradulent transaction representing very small fraction of total transactions.

## 4. EDA

In [ ]:
df['type'].value_counts()

In [ ]:
pd.crosstab(df['type'],df['isFraud'])

In [ ]:
pd.crosstab(df['type'],df['isFraud'],normalize='index')*100

In [ ]:
df['amount'].describe

In [ ]:
df.groupby('isFraud')['amount'].describe()

In [ ]:
(df['oldbalanceOrg'] - df['amount'] == df['newbalanceOrig']).value_counts()


## 5. Data preprocessing

In [ ]:
(df['oldbalanceDest'] + df['amount'] == df['newbalanceDest']).value_counts()


In [ ]:
pd.crosstab(df['isFlaggedFraud'], df['isFraud'])


In [ ]:
df.columns

In [ ]:
df = df.drop(columns=['nameOrig','nameDest'],errors='ignore')

In [ ]:
df = pd.get_dummies(df, columns=['type'], drop_first=True)


In [ ]:
df.isnull().sum()

In [ ]:
x = df.drop(columns=['isFraud','isFlaggedFraud'])
y = df['isFraud']

In [ ]:
from sklearn.model_selection import train_test_split
x_train,x_test,y_train,y_test = train_test_split(x,y,test_size=0.2,random_state=42,stratify=y)

In [ ]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()

x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

## 6. Model Building

In [ ]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(x_train_scaled,y_train)

## 7. Model Evaluation

In [ ]:
y_pred = model.predict(x_test_scaled)
y_pred_proba = model.predict_proba(x_test_scaled)[:,1]

In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test,y_pred))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test,y_pred)

In [ ]:
from sklearn.metrics import roc_auc_score
roc_auc_score(y_test,y_pred_proba)

## 8. Handling Class Imbalance

In [ ]:
from sklearn.linear_model import LogisticRegression

balanced_model = LogisticRegression(max_iter=1000, class_weight = 'balanced')
balanced_model.fit(x_train_scaled, y_train)

In [ ]:
y_pred_bal = balanced_model.predict(x_test_scaled)
y_pred_bal_proba = balanced_model.predict_proba(x_test_scaled)[:, 1]


In [ ]:
from sklearn.metrics import classification_report
print(classification_report(y_test, y_pred_bal))

In [ ]:
from sklearn.metrics import confusion_matrix
confusion_matrix(y_test, y_pred_bal)

In [ ]:
from sklearn.metrics import roc_auc_score

roc_auc_score(y_test, y_pred_bal_proba)


### Key Insights

- The dataset is highly imbalanced, with fraudulent transactions accounting for a very small percentage of total transactions.
- Fraud occurrences are concentrated primarily in specific transaction types, particularly TRANSFER and CASH_OUT.
- Rule-based fraud detection (`isFlaggedFraud`) captures only a small subset of actual fraudulent transactions.
- Incorporating class imbalance handling significantly improves the model’s ability to detect fraudulent transactions.

### Bussiness Recommendations

- Deploy machine learning–based fraud detection models alongside existing rule-based systems to improve fraud recall.
- Apply stricter monitoring and additional verification for high-risk transaction types such as TRANSFER and CASH_OUT.
- Continuously retrain the model using recent transaction data to adapt to evolving fraud patterns.
- Use model predictions to prioritize transactions for manual review rather than blocking all flagged transactions.